# Hosted Agent Session Files Upload/Download — `@azure/ai-projects`

This notebook demonstrates how to upload, list, download, and delete files within an agent Session using the `AIProjectClient`.

Sessions only work with Hosted Agents. It uploads the `basic-agent` code zip as a temporary Hosted Agent version, creates a session, then exercises the session file operations against that session.

It mirrors the [`sessionsFilesUploadDownload.ts`](./sessionsFilesUploadDownload.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`, `FOUNDRY_HOSTED_AGENT_NAME` (optional; defaults to `MyHostedAgent`).

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  CreateAgentVersionFromCodeContent,
  HostedAgentDefinition,
  VersionRefIndicator,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";
import { createHash } from "node:crypto";
import { readFileSync } from "node:fs";
import path from "node:path";
import { buffer } from "node:stream/consumers";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const modelName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
const agentName = process.env["FOUNDRY_HOSTED_AGENT_NAME"] ?? "MyHostedAgent";

const assetsDir = path.resolve("../assets");
const codeZipPath = path.join(assetsDir, "basic-agent.zip");
const dataFile1 = path.join(assetsDir, "data_file1.txt");
const dataFile2 = path.join(assetsDir, "data_file2.txt");
const remoteFilePath1 = "/remote/data_file1.txt";
const remoteFilePath2 = "/remote/data_file2.txt";

function sha256Hex(data: Uint8Array): string {
  return createHash("sha256").update(data).digest("hex");
}

console.log(`Model: ${modelName}`);
console.log(`Agent: ${agentName}`);

Model: gpt-5.2
Agent: MyHostedAgent9
Agent: MyHostedAgent9


In [2]:
// Create the AI Project client
// Annotated as `any` so tslab does not try to emit non-portable declarations
// referencing the deep `node_modules/openai` (pnpm junction) path.
const project: any = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [3]:
// Create a temporary hosted agent version from code
const codeZip = readFileSync(codeZipPath);
const codeZipSha256 = sha256Hex(codeZip);

const definition: HostedAgentDefinition = {
  kind: "hosted",
  cpu: "0.5",
  memory: "1Gi",
  protocol_versions: [{ protocol: "responses", version: "2.0.0" }],
  code_configuration: {
    runtime: "python_3_14",
    entry_point: ["python", "main.py"],
    dependency_resolution: "remote_build",
  },
  environment_variables: {
    FOUNDRY_PROJECT_ENDPOINT: projectEndpoint,
    FOUNDRY_MODEL_NAME: modelName,
  },
};

const content: CreateAgentVersionFromCodeContent = {
  metadata: {
    description: "Session files hosted agent uploaded from assets/basic-agent.",
    definition,
  },
  code: { contents: codeZip, contentType: "application/zip", filename: "code.zip" },
};

console.log("Creating hosted agent version from code...");
const created = await project.agents.createVersionFromCode(agentName, codeZipSha256, content);
const createdVersion = created.version;
console.log(`Created code-based hosted agent version: ${createdVersion}`);

Creating hosted agent version from code...
Created code-based hosted agent version: 121


In [4]:
// Poll until the agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, createdVersion);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1}/60)`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: creating (attempt 1/60)
Agent version status: creating (attempt 2/60)
Agent version status: creating (attempt 3/60)
Agent version status: creating (attempt 4/60)
Agent version status: creating (attempt 5/60)
Agent version status: creating (attempt 6/60)
Agent version status: creating (attempt 7/60)
Agent version status: creating (attempt 8/60)
Agent version status: creating (attempt 9/60)
Agent version status: creating (attempt 10/60)
Agent version status: creating (attempt 11/60)
Agent version status: creating (attempt 12/60)
Agent version status: active (attempt 13/60)


In [5]:
// Create a session
const versionIndicator: VersionRefIndicator = {
  type: "version_ref",
  agent_version: createdVersion,
};
const session = await project.agents.createSession(agentName, versionIndicator);
console.log(`Session created (id: ${session.agent_session_id}, status: ${session.status})`);

Session created (id: f56bc35057e0dc8100cyzOXo2rYf0cHltJDrWxcNTenNyGsY4q, status: active)


In [6]:
// Upload session files
console.log(`Uploading session file: ${dataFile1} -> ${remoteFilePath1}`);
await project.agents.uploadSessionFile(
  agentName,
  session.agent_session_id,
  remoteFilePath1,
  readFileSync(dataFile1),
);

console.log(`Uploading session file: ${dataFile2} -> ${remoteFilePath2}`);
await project.agents.uploadSessionFile(
  agentName,
  session.agent_session_id,
  remoteFilePath2,
  readFileSync(dataFile2),
);
console.log("Uploaded session files.");

Uploading session file: c:\Users\bogou\projects\azure-sdk-for-js\sdk\ai\ai-projects\samples-dev\agents\assets\data_file1.txt -> /remote/data_file1.txt
Uploading session file: c:\Users\bogou\projects\azure-sdk-for-js\sdk\ai\ai-projects\samples-dev\agents\assets\data_file2.txt -> /remote/data_file2.txt
Uploaded session files.


In [7]:
// List session files at path '/remote'
console.log("Listing session files for the session at path '/remote'...");
const listSessionFiles = async () => {
  const files = project.agents.listSessionFiles(agentName, session.agent_session_id, {
    path: "/remote",
  });
  for await (const entry of files) {
    console.log(`  - name=${entry.name}, size=${entry.size}, is_directory=${entry.is_directory}`);
  }
};
await listSessionFiles();

Listing session files for the session at path '/remote'...
  - name=data_file2.txt, size=22, is_directory=false
  - name=data_file1.txt, size=22, is_directory=false


In [8]:
// Download and print a session file
console.log(`Downloading and printing content from '${remoteFilePath1}'`);
const downloadResult = await project.agents.downloadSessionFile(
  agentName,
  session.agent_session_id,
  remoteFilePath1,
);
const fileContent = (await buffer(downloadResult.readableStreamBody!)).toString("utf-8");
console.log(`Session file content (${remoteFilePath1}):\n${fileContent}`);

Session file content (/remote/data_file1.txt):
This is sample file 1



In [9]:
// Delete session files
console.log(`Deleting session file at path: ${remoteFilePath1}...`);
await project.agents.deleteSessionFile(agentName, session.agent_session_id, remoteFilePath1);

console.log(`Deleting session file at path: ${remoteFilePath2}...`);
await project.agents.deleteSessionFile(agentName, session.agent_session_id, remoteFilePath2);
console.log("Deleted session files.");

Deleting session file at path: /remote/data_file1.txt...
Deleting session file at path: /remote/data_file2.txt...
Deleted session files.


In [10]:
// Clean up the session and the temporary agent version
await project.agents.deleteSession(agentName, session.agent_session_id);
console.log(`Session deleted (id: ${session.agent_session_id})`);

await project.agents.deleteVersion(agentName, createdVersion, { force: true });
console.log(`Agent version ${createdVersion} deleted.`);

Session deleted (id: f56bc35057e0dc8100cyzOXo2rYf0cHltJDrWxcNTenNyGsY4q)
Agent version 121 deleted.
